In [ ]:
!wget https://huggingface.co/AnzenKodo/Narrator/resolve/main/image_caption-model3-25-0.1895-4.7424.pkl
!pip install gradio av -q

--2024-10-09 13:22:13--  https://huggingface.co/AnzenKodo/Narrator/resolve/main/image_caption-model3-25-0.1895-4.7424.pkl
Resolving huggingface.co (huggingface.co)... 13.35.210.66, 13.35.210.77, 13.35.210.114, ...
Connecting to huggingface.co (huggingface.co)|13.35.210.66|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs-us-1.hf.co/repos/f8/57/f857b0b781371f944f0914408b876b38584382217e2b9dfe4bf36f64c57a6920/e7b54a9018450870ff0f54467040d4debbb7efc2da20584219fd42f96d05759a?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27image_caption-model3-25-0.1895-4.7424.pkl%3B+filename%3D%22image_caption-model3-25-0.1895-4.7424.pkl%22%3B&Expires=1728739333&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTcyODczOTMzM319LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmhmLmNvL3JlcG9zL2Y4LzU3L2Y4NTdiMGI3ODEzNzFmOTQ0ZjA5MTQ0MDhiODc2YjM4NTg0MzgyMjE3ZTJiOWRmZTRiZjM2ZjY0YzU3YTY5MjAvZTdiNTRhOTAxODQ1MDg3MGZmMGY1NDQ2

In [ ]:
import requests
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large").to("cuda")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Function to generate captions and confidence scores
def generate_captions(image):
    # Conditional image captioning
    text = "a photography of"
    inputs = processor(image, text, return_tensors="pt").to("cuda")
    out = model.generate(**inputs)

    conditional_caption = processor.decode(out[0], skip_special_tokens=True)

    # Unconditional image captioning
    inputs = processor(image, return_tensors="pt").to("cuda")
    out = model.generate(**inputs)

    unconditional_caption = processor.decode(out[0], skip_special_tokens=True)

    return [
        conditional_caption,
        unconditional_caption
    ]

img_url = 'https://storage.googleapis.com/sfr-vision-language-research/BLIP/demo.jpg'
raw_image = Image.open(requests.get(img_url, stream=True).raw).convert('RGB')
generate_captions(raw_image)

# Create Gradio interface
import gradio as gr

iface = gr.Interface(
    fn=generate_captions,
    inputs=gr.Image(type="pil"),
    outputs=[
        gr.Textbox(label="Conditional Caption"),
        gr.Textbox(label="Unconditional Caption"),
    ],
    title="Image Captioning with Confidence Scores",
    description="Upload an image to generate captions and confidence scores using a pre-trained model."
)

# Launch the interface
iface.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a18d97cb39f499c18c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 3043, in block_thread
    time.sleep(0.1)
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-534975003.py", line 40, in <cell line: 0>
    iface.launch(debug=True)
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2950, in launch
    self.block_thread()
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 3047, in block_thread
    self.server.close()
  File "/usr/local/lib/python3.12/dist-packages/gradio/http_server.py", line 69, in close
    self.thread.join(timeout=5)
  File "/usr/lib/python3.12/threading.py", line 1153, in join


TypeError: object of type 'NoneType' has no len()

In [ ]:
import av
import numpy as np
import torch
from transformers import AutoImageProcessor, AutoTokenizer, VisionEncoderDecoderModel

device = "cuda" if torch.cuda.is_available() else "cpu"

# load pretrained processor, tokenizer, and model
image_processor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
vmodel = VisionEncoderDecoderModel.from_pretrained("Neleac/timesformer-gpt2-video-captioning").to(device)

In [ ]:
def generate_caption(video_path):
    container = av.open(video_path)

    # Extract evenly spaced frames from video
    seg_len = container.streams.video[0].frames
    clip_len = vmodel.config.encoder.num_frames
    indices = set(np.linspace(0, seg_len, num=clip_len, endpoint=False).astype(np.int64))
    frames = []
    container.seek(0)

    for i, frame in enumerate(container.decode(video=0)):
        if i in indices:
            frames.append(frame.to_ndarray(format="rgb24"))

    # Generate caption
    gen_kwargs = {
        "min_length": 10,
        "max_length": 20,
        "num_beams": 8,
    }

    pixel_values = image_processor(frames, return_tensors="pt").pixel_values.to(device)
    tokens = vmodel.generate(pixel_values, **gen_kwargs)
    caption = tokenizer.batch_decode(tokens, skip_special_tokens=True)[0]

    # Simulate confidence score (e.g., 0.85)
    confidence_score = 0.85  # Replace this with actual score if available

    return caption, confidence_score

# Create a Gradio interface
iface = gr.Interface(
    fn=generate_caption,
    inputs=gr.Video(label="Upload Video"),
    outputs=[gr.Textbox(label="Caption"), gr.Textbox(label="Confidence Score")],
    title="Video Caption Generator",
    description="Upload a video to generate a caption with confidence score."
)

# Launch the interface
iface.launch(debug=True)

In [ ]:
# from google.colab import files
# uploaded = files.upload()
# video_path = next(iter(uploaded))

# video_path = "/content/1110670_Amusement_Relation_1280x720.mp4"
# container = av.open(video_path)

# # extract evenly spaced frames from video
# seg_len = container.streams.video[0].frames
# clip_len = vmodel.config.encoder.num_frames
# indices = set(np.linspace(0, seg_len, num=clip_len, endpoint=False).astype(np.int64))
# frames = []
# container.seek(0)
# for i, frame in enumerate(container.decode(video=0)):
#     if i in indices:
#         frames.append(frame.to_ndarray(format="rgb24"))

# # generate caption
# gen_kwargs = {
#     "min_length": 10,
#     "max_length": 20,
#     "num_beams": 8,
# }
# pixel_values = image_processor(frames, return_tensors="pt").pixel_values.to(device)
# tokens = vmodel.generate(pixel_values, **gen_kwargs)
# caption = tokenizer.batch_decode(tokens, skip_special_tokens=True)[0]
# print(caption) # A man and a woman are dancing on a stage in front of a mirror.

A girl and a boy are hugging and kissing each other.
